In [3]:
!setenv CUDA_VISIBLE_DEVICES 3

In [4]:
from bioservices import Ensembl
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import sys
import json

In [5]:
ensembl = Ensembl()
genes = pd.read_csv('/local/home/am/EX_M/datasets/scRNA/merged/genes.csv', index_col=0)


def fetch_data(symbols):
    try:
        response = ensembl.post_lookup_by_symbol(symbols=symbols, species="homo_sapiens", expand=True)
        return response, symbols
    except Exception as e:
        print(f"Failed to fetch data for {symbols}: {e}")
        return None

# Function to split gene list into chunks
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# Number of symbols per thread
symbols_per_thread = 30

# Total number of threads
num_workers = 10

# This function manages the distribution of gene symbols across threads and monitors progress
def process_genes(gene_symbols):
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        # Create a list of tasks submitted to the executor
        tasks = [executor.submit(fetch_data, chunk) for chunk in chunks(gene_symbols, symbols_per_thread)]
        all_responses = {}

        # Use tqdm to track the progress of tasks
        for task in tqdm(as_completed(tasks), total=len(tasks), desc="Processing Genes", unit="batch"):
            try:
                response, _ = task.result()
                if response:
                    all_responses.update(response)
            except Exception as e:
                print(f"Error processing task: {e}")

        return all_responses

# Process all gene symbols
all_gene_responses = process_genes(genes.index.tolist())

Creating directory /cs/home/am/.cache/bioservices 
Welcome to Bioservices
It looks like you do not have a configuration file.
We are creating one with default values in /cs/home/am/.config/bioservices/bioservices.cfg .
Done


Processing Genes: 100%|██████████| 558/558 [08:25<00:00,  1.10batch/s]


In [7]:
all_gene_responses['AGRN']

{'strand': 1,
 'assembly_name': 'GRCh38',
 'display_name': 'AGRN',
 'canonical_transcript': 'ENST00000379370.7',
 'species': 'homo_sapiens',
 'Transcript': [{'version': 7,
   'end': 1056116,
   'Parent': 'ENSG00000188157',
   'object_type': 'Transcript',
   'source': 'ensembl_havana',
   'biotype': 'protein_coding',
   'seq_region_name': '1',
   'is_canonical': 1,
   'Exon': [{'species': 'homo_sapiens',
     'version': 2,
     'end': 1020373,
     'strand': 1,
     'assembly_name': 'GRCh38',
     'object_type': 'Exon',
     'id': 'ENSE00002277560',
     'db_type': 'core',
     'start': 1020120,
     'seq_region_name': '1'},
    {'end': 1022462,
     'strand': 1,
     'assembly_name': 'GRCh38',
     'version': 1,
     'species': 'homo_sapiens',
     'start': 1022201,
     'seq_region_name': '1',
     'db_type': 'core',
     'id': 'ENSE00001673004',
     'object_type': 'Exon'},
    {'db_type': 'core',
     'id': 'ENSE00003635914',
     'object_type': 'Exon',
     'start': 1035277,
     '